# Building a cable object

In this example we will discuss how to use the `CableBuilder` class to create `FDCable` objects. There are two ways to do this.

## 1. Use the `build_cable_from_cable_id` method

This method reads the constructional information from a `.csv` or `.xlsx` file and builds the cable.

A couple of example cables are included in the `data/example_cables.csv` file. This file contains all the properties that are needed to construct a cable object, including
- Materials used in the cable layers.
- Dimensions of the cable layers.
- Additional information, such as the conductor shape and nominal voltage.

The column structure of the file equals the column structure of cable exports from [Vision Cable Analysis](https://www.phasetophase.nl/producten-diensten/producten/vision-cable-analysis/), developed by [Phase to Phase](https://www.phasetophase.nl/).

In [ ]:
from pathlib import Path

from cable_thermal_model.cable.cable_builder import CableBuilder
from cable_thermal_model.model.cables.cable import CableSoil

cable = CableBuilder.build_cable_from_cable_id(
    cable_id="YMeKrvaslqwd 12/20kV 1x630 Alrm + as50",
    cable_class=CableSoil,
    cable_source_file_path=Path("../../data/example_cables.csv"),
)

Usually a circuit is added directly to a static environment by using the `add_circuit_from_cable_id` method. The cable is then automatically built and added to the circuit. Again, a path to a cable source file can be specified.

In [ ]:
from cable_thermal_model import BondingType, CircuitType, StaticEnvSoil
from cable_thermal_model.cable.schemas.circuit_schemas import (
    CircuitInSoilFromCableIdInputSchema,
)

static_env = StaticEnvSoil()

static_env.add_circuit_from_cable_id(
    CircuitInSoilFromCableIdInputSchema(
        x=0.0,
        y=-0.7,
        circuit_name="Circuit 1",
        cable_id="YMeKrvaslqwd 12/20kV 1x630 Alrm + as50",
        cable_source_file_path=Path("../../data/example_cables.csv"),
        bonding_type=BondingType.TwoSided,
        circuit_type=CircuitType.Trefoil,
    )
)

StaticEnvSoil
	Circuit Circuit 1
		Type: TrefoilCircuit
		BondingType: two_sided
		x: 0.0
		y: -0.7

## 2. Use the `build_cable` method

This method reads the constructional information of the cable in the form of a `CableConstructionalInputSchema`. This schema can be completed manually by first constructing the separate layers and then specifying the number of conductors. Each cable layer is validated individually. When the layers are assembled in `CableConstructionalInputSchema` additional validation is performed to check if the provided layers are consistent. 

> **_NOTE:_** This validation also happens when using option 1. In option 1, a `CableSpecParser` is used to fill a `CableConstructionalInputSchema`.

The most simple possible cable consists of a conductor, insulation and a sheath. Other layers that can be used to construct a cable include for example internal oil ducts and conductor screen layers.

Cable layers that can be used are listed below. The layers are always to assume to appear in the listed order starting at the center of the cable. With `CableLayer.Screen`, the earthing screen or sheath is meant, this may be confusing with `CableLayer.Sheath` which depicts the outer cable sheath or serving. 

- Optional: `CableLayer.InnerOilDuct`
- Mandatory: `CableLayer.Conductor`
- Optional: `CableLayer.ConductorScreen`
- Mandatory: `CableLayer.Insulation`
- Optional: `CableLayer.InsulationScreen`
- Optional: `CableLayer.Screen`
- Optional: `CableLayer.Bedding`
- Optional: `CableLayer.Armour`
- Mandatory: `CableLayer.Sheath`


In [ ]:
from cable_thermal_model.cable.schemas.cable_input_schemas import (
    CableConstructionalInputSchema,
)
from cable_thermal_model.cable.schemas.cable_layer_input_schemas import (
    ConductorInputSchema,
    InsulationInputSchema,
    SheathInputSchema,
)
from cable_thermal_model.model.cables.enum_classes_cable import (
    CableConductorCount,
    CableConductorShape,
    CableConductorSurfaceType,
    CableInsulationMaterial,
    CableSheathMaterial,
)

conductor = ConductorInputSchema(
    material="cmAl",
    inner_radius=0.0,
    outer_radius=0.01395,
    conducting_surface_area=6.30e-4,
    shape=CableConductorShape.Round,
    surface_type=CableConductorSurfaceType.Stranded,
)

insulation = InsulationInputSchema(
    material=CableInsulationMaterial.XLPEUnfilled,
    thickness=0.010,
    nominal_phase_voltage=12000.0,
)

sheath = SheathInputSchema(
    material=CableSheathMaterial.PE,
    thickness=0.005,
)

cable_constructional_input = CableConstructionalInputSchema(
    number_of_conductors=CableConductorCount.One,
    conductor_input=conductor,
    insulation_input=insulation,
    sheath_input=sheath,
)

simple_cable = CableBuilder.build_cable(
    cable_constructional_input=cable_constructional_input,
    cable_class=CableSoil,
)

The cable instance can then be added to a static environment using the `add_circuit_from_cable` method.

In [ ]:
from cable_thermal_model.cable.schemas.circuit_schemas import (
    CircuitInSoilFromCableInputSchema,
)

static_env.add_circuit_from_cable(
    CircuitInSoilFromCableInputSchema(
        x=0.5,
        y=-0.7,
        circuit_name="Circuit 2",
        cable=simple_cable,
        bonding_type=BondingType.NoBonding,
        circuit_type=CircuitType.Trefoil,
    )
)
static_env

StaticEnvSoil
	Circuit Circuit 1
		Type: TrefoilCircuit
		BondingType: two_sided
		x: 0.0
		y: -0.7
	Circuit Circuit 2
		Type: TrefoilCircuit
		BondingType: no_bonding
		x: 0.5
		y: -0.7